# Anwenderwebsite lokal bauen

Dieses How-to baut dieselben Lehrgrafiken, Downloads und Notebook-Lesefassungen wie Quarto. Die realen Beispiele werden aus `website/_examples.yml` ausgewählt. Voraussetzung: `python -m pip install -e ".[publication]"` im Repository, Quarto für die HTML-Seiten. Für die vorhandenen synthetischen Grafikexporte zusätzlich Chrome/Kaleido.

Die Website läuft statisch, beispielsweise auf GitHub Pages. Sie nutzt das bestehende Repository und Python-Paket beim Build; der Browser führt keine Python- oder Excel-Simulation aus. Eine Veröffentlichung ist ein separater Schritt.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts/website_assets.py").exists()
sys.path.insert(0, str(ROOT / "scripts"))
from website_assets import build
output = build(ROOT / "notebooks/local_exports/website_tutorial")
assert (output / "assets/teaching/index.html").exists()
assert (output / "assets/components/index.html").exists()
print("Lehrmodule, öffentliche Unterlagen und Notebook-Lesefassungen erstellt.")

## Lehrgrafiken ausprobieren

Im ersten Modul: Bauteil auswählen, Hülle aktivieren, gekühlte/ungekühlte Zone vergleichen, einzelne Wärmeflüsse ausblenden, aktuellen Zustand als SVG speichern. Der zweite Schnitt zeigt eine ausdrücklich angenommene unbeheizte Kellerzone. Die Kategorienamen stammen aus dem bestehenden Schema; nicht vorhandene Elemente erhalten keine Ersatzzuordnung.

In [ ]:
from IPython.display import HTML, display
from component_teaching import categories
assert len(categories()) == 20
display(HTML('<iframe title="Bauteilkatalog" style="width:100%;height:850px" srcdoc="' + __import__('html').escape((output / "assets/components/index.html").read_text(encoding="utf-8"), quote=True) + '"></iframe>'))

## Quarto rendern und prüfen

Im Repository: `quarto render website`. Das deutsche Projekt baut die Hauptseiten; sein Post-Render-Schritt startet das englische Unterprojekt `website/en`. Beide Sprachfassungen landen gemeinsam in `website/_site`, die englischen Seiten unter `_site/en/`.

Die beiden `_quarto.yml`-Dateien definieren getrennte Kopf- und Seitenleisten. Die Quelldateien liegen im selben Sprachverzeichnis wie ihre Navigation. Beide Versionen lokal über die Website-Vorschau kontrollieren; nach Inhaltsänderungen den Build erneut ausführen.

Die Notebook-Lesefassungen und Downloads enthalten keine gespeicherten Zellenausgaben. Die Originale bleiben unverändert. Quellen: `docs/`, `notebooks/`, `data/schemas/1.13.1_dev/IN.csv`. Generierte Veröffentlichungsdateien liegen in `website/_site`.

## Konfigurierte Ergebnisse erzeugen

Website und Publikations-Notebook verwenden denselben Adapter. Hier werden die interaktiven HTML-Ergebnisse ohne erneute statische Bildausgabe erzeugt.

In [ ]:
from website_examples import build as build_examples
reports, checks = build_examples(output / "assets/results", formats=("html",))
assert len(reports) == 4
checks

## Excel-Dokumentation und bereitgestellte Abbildungen

Die deutsche Navigation trennt Tool-Anwendung, Modellieren und Analysieren und Bewerten. Die englische Quarto-Unterseite definiert dieselbe Struktur mit eigener englischer Kopf- und Seitenleiste. `results.qmd` beschreibt Erg; `excel-dashboard.qmd` beschreibt das Excel-Dashboard; `dashboard.qmd` bleibt der Python-Szenarioexplorer. Migration, PV, Varianten und Scen besitzen eigene Seiten.

Originalbilder liegen unter `docs/website/`. Die explizite Liste `MANUAL_FIGURES` in `website_assets.py` ist die Veröffentlichungsauswahl. `build` kopiert sie unverändert nach `media/manual/`; keine veralteten Handbuch-Screenshots werden erzeugt. Bei Austausch Quellen-/Versionshinweis in der QMD-Bildunterschrift mitpflegen.

Bedienorte mit `.excel-menu` markieren, Pfade in `.excel-path`, Schritte in `.excel-operation`, Felder mit `.excel-field`. Fachliche Erläuterungen bleiben normaler Fließtext; Nachweisbindungen stehen in `.declaration-note`. Jede Abbildung benötigt Alt-Text und eine Quellenangabe. Die Marken sind keine Excel-Schaltflächen.

Beispiel: `**Excel-Ort:** Blatt [🔥 Heizung]{.excel-menu .darkred}`. Die vollständige Formatierungslegende steht in `excel-navigation.qmd`.


In [ ]:
from website_assets import MANUAL_FIGURES
for name in MANUAL_FIGURES:
    source = ROOT / "docs/website" / f"{name}.png"
    published = output / "media/manual" / source.name
    assert published.read_bytes() == source.read_bytes()
print(f"{len(MANUAL_FIGURES)} Originalabbildungen unverändert veröffentlicht.")


### Visuelle Abnahme

Nach `quarto render website` die Seiten Presets, Blattnavigator und Excel-Dashboard auf Desktop und bei 390 px Breite prüfen. Blattmarken und Pfade müssen umbrechen; Bilder sind auf weißem Hintergrund und per Lightbox vergrößerbar. Fehlende Ansichten sind als „In Vorbereitung“ gekennzeichnet. Linkziele und Anker einschließlich F7 unter `examples.qmd#aspern-f7` prüfen.


## English pages

English mirrors live in `website/en/` as a separate Quarto website project with `lang: en`. It defines its own English navbar and sidebar, while the German project keeps its German navigation. Keep German worksheet and ribbon labels where they identify controls in the workbook. English links point to pages within `en/`; images, downloads and generated visualisations use paths relative to the parent website. Running `quarto render website` renders the German site and then the English subproject into the shared `_site/en/` output. The English render is launched with `website/en` as its working directory so Quarto reads the English project configuration.
